In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
from glob import glob
from os.path import join as opj
import pandas as pd
import gzip
from tqdm import tqdm

In [ ]:
import torchmetrics
from torchmetrics.functional.classification import accuracy
import logging

In [ ]:
import altair as alt

In [ ]:
def load_data(fn, perturb_width):
    with gzip.open(fn) as fd:
        data = torch.load(fd)

    return pd.DataFrame({
        "path": data["path"],
        "embeddings":  data["embeddings"].tolist(),
        "perturb_width": perturb_width,
        "logits": data["logits"].tolist(),
        "label": data["label"]
    })

In [ ]:
def load_data_raw(fn, perturb_width):
    with gzip.open(fn) as fd:
        data = torch.load(fd)

    return data
def make_pred_df(data, perturb_width):
    return pd.DataFrame({
        "path": data["path"],
        "embeddings":  data["embeddings"].tolist(),
        "perturb_width": perturb_width,
        "logits": data["logits"].tolist(),
        "label": data["label"]
    })

In [ ]:
def batched_heatmap_to_rgb(heatmaps: np.ndarray) -> np.ndarray:
    """
    Convert batched heatmaps (values in [0,1]) to RGB using viridis colormap.
    
    Args:
        heatmaps: np.ndarray of shape (B, L, A, H, W)

    Returns:
        np.ndarray of shape (B, L, A, H, W, 3), dtype uint8
    """
    cmap = plt.get_cmap('viridis')
    B, L, A, H, W = heatmaps.shape
    flat = heatmaps.reshape(-1, H, W)
    rgba = cmap(flat)               # (B*L*A, H, W, 4)
    rgb = (rgba[..., :3] * 255).astype(np.uint8)
    return rgb.reshape(B, L, A, H, W, 3)

In [ ]:
def nice_chart(chart, chart_size=400):
    return chart.properties(
        width=chart_size, height=chart_size
    ).configure_axis(
        tickSize=0,
        labelFontSize=16,
        titleFontSize=16
    ).configure_legend(
        labelFontSize=16,
        titleFontSize=16
    ).configure_title(
        fontSize=16,
    )

In [ ]:
def binary_neg_vs_rest_accuracy(probs, labels, negative_class=4):
    bin_labels = (labels == negative_class).long()

    bin_preds = probs[:, negative_class]

    acc = torchmetrics.functional.classification.accuracy(
        bin_preds,
        bin_labels,
        task="binary",
        average="macro"
    )
    
    
    return acc


In [ ]:
from ts2.eval.common import get_knn_logits_exclude_slide, knn_predict

In [ ]:
knn_cf = {
    "data":{
        "test_dataset": {
            "classes_reorder": [None] * 7  
        }
    },
    "testing":{
        "knn":{
            "knn_params":{
                "batch_size": 1536,
                "k": 20,
                "t": 0.07
            }
        }
    }
}

# Main

## Find and load predictions

In [ ]:
def get_results_one_eval(exp_name, ckpt_iter, eval_regex, eval_prefix1="CBSP"):
    preds = pd.DataFrame(glob(f"/nfs/turbo/umms-tocho-snr/exp/chengjia/ts2/fmi_dinov2_cc_new/{exp_name}/models/eval/{ckpt_iter}/{eval_regex}/predictions/val_predictions.pt.gz"), columns=["pred_fn"])
    preds["band_width"] = preds["pred_fn"].apply(lambda x:int(x.split("/")[-3].split("_")[5].removeprefix(eval_prefix1)))
    preds = preds.sort_values("band_width").reset_index(drop=True)

    no_perturb_train_preds = preds.loc[preds["band_width"]==0, "pred_fn"]
    assert(len(no_perturb_train_preds)==1)

    with gzip.open(no_perturb_train_preds.iloc[0]) as fd:
        train_data_unperturbed = torch.load(fd)

    # pred_data = [load_data_raw(i["pred_fn"], i["band_width"]) for _,i in preds.iterrows()]
    pred_data = [load_data(i["pred_fn"], i["band_width"]) for _,i in preds.iterrows()]

    metrics = {
        "srh7": torch.stack([
            accuracy(torch.tensor(i["logits"]),torch.tensor(i["label"]), 
                     task="multiclass", average="macro", num_classes=7)
            for i in pred_data
        ]),
        "srh7_tn": torch.stack([
            binary_neg_vs_rest_accuracy(torch.tensor(i["logits"]), torch.tensor(i["label"]))
            for i in pred_data
        ]),
        "band_width": preds["band_width"],
    }
    return metrics

In [ ]:
ckpt_iter = "training_124999"

eval_configurations = {
    "89d3ad98_May23-13-58-49_sd1000_dev_tune0": [
        "*BENCHDB_CBSP*_epoch0-step124999_tune*",
        #"*BENCHDB_CBSP*_epoch0-step124999_sd2k*",
        #"*BENCHDB_CBSP*_epoch0-step124999_sd3k*"
    ],
    "6778e5d1_May27-15-59-58_sd1000_dev_tune0": [
        "*BENCHDB_CBSP*_epoch0-step124999_tune*",
        #"*BENCHDB_CBSP*_epoch0-step124999_sd2k*",
        #"*BENCHDB_CBSP*_epoch0-step124999_sd3k*"
    ],
    "a1ceb84e_Jan10-05-16-51_sd1000_newrun_dev_maskobw_tune1":[
        "*BENCHDB_CBSP*_epoch0-step124999_sd1k*",
    ],
    "546d5129_Jan12-20-08-19_sd1000_newrun_dev_nomaskobw_lr54_tune0":[
        "*BENCHDB_CBSP*_epoch0-step124999_sd1k*",
    ],
    "c2b59e45_Jan12-20-08-19_sd1000_newrun_dev_maskobw_lr54_tune1": [
         "*BENCHDB_CBSP*_epoch0-step124999_sd1k*",
    ]
}
    
#model_name = exp_name.split("_")[0]

In [ ]:
exps = list(eval_configurations.keys())
means = []
stds = []
band_widths = None
for k in exps:
    all_results = []
    for ev in tqdm(eval_configurations[k]):
        all_results.append(get_results_one_eval(k, ckpt_iter, ev))
        if band_widths == None:
            band_widths = torch.tensor(all_results[-1]["band_width"])
        else:
            assert (band_widths ==  torch.tensor(all_results[-1]["band_width"])).all()
            
    means.append(torch.mean(torch.stack([x["srh7"] for x in all_results]), axis=0))
    stds.append(torch.std(torch.stack([x["srh7"] for x in all_results]), axis=0))

In [ ]:
perct_masked = band_widths * 10

E = len(exps)
L = len(perct_masked)

# (E, L)
mean_mat = torch.stack(means).cpu().numpy()
std_mat  = torch.stack(stds).cpu().numpy()

# (E, L)
lower = mean_mat - std_mat
upper = mean_mat + std_mat

# tile and repeat for broadcasted assembly (no loops)
df = pd.DataFrame({
    "method":       np.repeat(exps, L),
    "perct_masked": np.tile(perct_masked, E),
    "mca":          mean_mat.reshape(-1),
    "std":          std_mat.reshape(-1),
    "lower":        lower.reshape(-1),
    "upper":        upper.reshape(-1),
})

In [ ]:
df["method"] = df["method"].map({
    "89d3ad98_May23-13-58-49_sd1000_dev_tune0": "DINOv2",
    "6778e5d1_May27-15-59-58_sd1000_dev_tune0": "Ours",
    "a1ceb84e_Jan10-05-16-51_sd1000_newrun_dev_maskobw_tune1": "Inside iBOT mask",
    "546d5129_Jan12-20-08-19_sd1000_newrun_dev_nomaskobw_lr54_tune0": "Ours_lr54",
    "c2b59e45_Jan12-20-08-19_sd1000_newrun_dev_maskobw_lr54_tune1": "Inside_lr54",
})

In [ ]:
band = (
    alt.Chart(df)
    .mark_area(opacity=0.15)
    .encode(
        x=alt.X("perct_masked:Q", title="Perct Masked"),
        y=alt.Y("lower:Q", title="MCA"),
        y2="upper:Q",
        color="method:N",
    )
)

line = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x="perct_masked:Q",
        y="mca:Q",
        color="method:N",
    )
)

nice_chart(band + line).interactive()

## Classification metrics

In [ ]:
#pred_data_unperturb_knn = [
#    make_pred_df(get_knn_logits_exclude_slide(knn_cf, train_data_unperturbed, pdi), i["band_width"])
#    for pdi, (_, i) in tqdm(zip(pred_data, preds.iterrows()))
#]

In [ ]:
torch.stack([torchmetrics.functional.classification.accuracy(torch.tensor(i["logits"]), torch.tensor(i["label"]), task="multiclass", average="macro", num_classes=7)
for i in pred_data])

In [ ]:
torch.stack([binary_neg_vs_rest_accuracy(torch.tensor(i["logits"]), torch.tensor(i["label"]))
for i in pred_data])

In [ ]:
torch.stack([torchmetrics.functional.classification.accuracy(torch.tensor(i["logits"]), torch.tensor(i["label"]), task="multiclass", average="macro", num_classes=7)
for i in pred_data])

In [ ]:
torch.stack([binary_neg_vs_rest_accuracy(torch.tensor(i["logits"]), torch.tensor(i["label"]))
for i in pred_data])

In [ ]:
import altair as alt

In [ ]:
dinov2 = np.array([0.9483, 0.9361, 0.9210, 0.8940, 0.8654, 0.8442, 0.8144, 0.7725, 0.7530,
        0.7300, 0.7144])
ours =   np.array([0.9551, 0.9424, 0.9410, 0.9293, 0.9160, 0.9028, 0.8840, 0.8496, 0.8315,
        0.7719, 0.7090])
nice_chart(alt.Chart(pd.melt(pd.DataFrame({
    "perct_masked": preds["band_width"]*10,
    "dinov2": dinov2,
    "ours": ours,
}), id_vars="perct_masked",
        value_vars=["dinov2", "ours"],
        var_name="method",
        value_name="mca")).mark_line(point=True).encode(
x="perct_masked", y="mca", color="method"))

In [ ]:
dinov2 = np.array([0.5062, 0.4810, 0.4537, 0.4027, 0.3530, 0.3120, 0.2634, 0.2067, 0.1878,
        0.1648, 0.1451])
ours =   np.array([0.5250, 0.4908, 0.4978, 0.4822, 0.4590, 0.4382, 0.3975, 0.3346, 0.3077,
        0.2150, 0.1452])
ours_new = np.array([0.5262, 0.4870, 0.4984, 0.4885, 0.4673, 0.4449, 0.4057, 0.3448, 0.3080,
        0.2089, 0.1457])
nice_chart(alt.Chart(pd.melt(pd.DataFrame({
    "perct_masked": preds["band_width"],
    "dinov2": dinov2,
    "ours": ours,
    "ours_new": ours_new,
}), id_vars="perct_masked",
        value_vars=["dinov2", "ours", "ours_new"],
        var_name="method",
        value_name="mca")).mark_line(point=True).encode(
x="perct_masked", y="mca", color="method"))

## Instance discrimination metrics

In [ ]:
all_pred_data = pd.concat([i.iloc[:256] for i in pred_data])
all_pred_data_sorted = all_pred_data.sort_values("path", kind="stable").reset_index(drop=True)
embs = torch.nn.functional.normalize(torch.tensor(all_pred_data_sorted["embeddings"]), dim=1)

In [ ]:
all_pred_data_sorted.keys()

In [ ]:
def heatmap_to_rgb(heatmap: np.ndarray, cmap_name='viridis') -> np.ndarray:
    cmap = plt.get_cmap(cmap_name)
    rgba_img = cmap(heatmap)  # shape: (H, W, 4), values in [0, 1]
    rgb_img = (rgba_img[..., :3] * 255).astype(np.uint8)  # drop alpha, convert to uint8
    return rgb_img

In [ ]:
sim3 = embs@embs.T
plt.imshow(mean3)
plt.clim(0, 1)
plt.colorbar()
#Image.fromarray(heatmap_to_rgb(sim3)).save("89d3ad98_perturb_all.png")

In [ ]:
mean3 = torch.mean(torch.stack([sim3[i*7:(i+1)*7, i*7:(i+1)*7] for i in range(256)]), dim=0)
#Image.fromarray(heatmap_to_rgb(mean3)).save("89d3ad98_perturb_mean.png")
plt.imshow(mean3)
plt.clim(0, 1)
plt.colorbar()

# Instance discriminatioin

In [ ]:
def get_acc(clean_df, query_df):
    clean = torch.nn.functional.normalize(torch.tensor(clean_df["embeddings"]), dim=1)
    query = torch.nn.functional.normalize(torch.tensor(query_df["embeddings"]), dim=1)
    acc = ((query@clean.T).argmax(dim=1) == torch.arange(len(clean))).sum() / len(clean)
    return acc.item()

In [ ]:
[get_acc(pred_data[0], pred_data[i]) for i in range(7)]

In [ ]:
df = pd.DataFrame({"DINOv2": [0.9906283617019653,
 0.8060129880905151,
 0.1594528704881668,
 0.013082340359687805,
 0.0045774648897349834,
 0.004171181004494429,
 0.004171181004494429],
      "Silica": [0.9906283617019653,
 0.9365384578704834,
 0.7224810123443604,
 0.33854278922080994,
 0.04534127935767174,
 0.0042253523133695126,
 0.004171181004494429], "perturb_width":np.arange(7)})

In [ ]:
df_melted = pd.melt(df, id_vars=['perturb_width'], value_vars=['DINOv2', "Silica"], var_name='method', value_name='acc')


In [ ]:
import altair as alt

In [ ]:
xaxis = alt.Axis(tickSize=0,
                          values=np.arange(7),
                          title="Perturbation width")
yaxis = alt.Axis(tickSize=0,
                 values=np.linspace(0,1,6),
                          title="Instance discrimination accuracy")
alt.Chart(df_melted).mark_line(point=True).encode(
    x=alt.X("perturb_width", axis=xaxis, scale=alt.Scale(domain=[0,6])),
    y=alt.Y("acc:Q",axis=yaxis),
    color=alt.Color('method', legend=alt.Legend(title="Method"))
).properties(width=400,height=400).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_legend(titleFontSize=14,labelFontSize=14).interactive()#.save("perturb_instance_discrimination.json")


# Get image sample

In [ ]:
from torchvision.transforms.functional import adjust_contrast, adjust_brightness
import einops
from PIL import Image

In [ ]:
ims = torch.stack([adjust_brightness(adjust_contrast(
    torch.load(f"../train/band_perturb{i}.pt")["alt_fg"],
    2), 3) for i in range(7)])

In [ ]:
ims_reshaped = einops.rearrange(ims, "p b c h w -> b p h w c")

In [ ]:
torch.save(ims_reshaped, "perturb_im_viz.pt")

In [ ]:
mkdir perturb_im_viz

In [ ]:
for ii, i in enumerate(ims_reshaped):
    for ij, j in enumerate(i):
        
        Image.fromarray((j*255).numpy().astype(np.uint8)).save(f"perturb_viz_{ii}_{ij}.png")

In [ ]:
for i in range(256):
    plt.figure()
    plt.imshow(einops.rearrange(ims_reshaped[i], "p h w c -> h (p w) c"))